# 10v5_03 — TL regime audit (no training)

**Questions this notebook measures (none of them is assumed):**

1. How often does the TRUE candidate of a TL_EVAL query have at least one eligible reference, under
   `standard` and under `mirror_aware`?
2. Does TL_EVAL mix Mode-A queries (truth has reference evidence) with reference-absent ones, and
   how does that compare with HOST?
3. Do test-like timsTOF spectra from Enveda-owned libraries lose their truth references mainly
   because `mirror_aware` excludes same-library evidence? *This is a question. It is answered
   by the source-library breakdown below.*

**Protocols vs the hidden test**

| protocol | allows | reading |
|---|---|---|
| `standard` | same-source / same-library references (only the query itself is excluded) | optimistic for a benchmark query whose library mates were measured in the same session |
| `mirror_aware` | no same-library references and no T1/T2 mirrors | pessimistic when the hidden test shares an instrument or platform, but not the session, with training libraries |
| hidden test | unpublished query spectra; training references may come from the same instrument or platform, never from the hidden query's own session | lies somewhere between the two; neither protocol is claimed to be the exact hidden-test regime |

So this audit reports a **diagnostic range**, not a test MRR forecast. It consumes existing QCR
evidence only (no similarity is recomputed, and nothing is trained). A dataset whose QCR has not
been built yet is reported as `NOT_BUILT`.

Regimes: `MODE_A_MIRROR`, `STANDARD_ONLY`, `REF_ABSENT`, `POOL_ABSENT`, and `INCONSISTENT`, which
should be empty by construction and is asserted. The routing rule is fixed before execution:
if the TL_EVAL `MODE_A_MIRROR` share is below **0.50**, the route is
`PRIORITIZE_REFERENCE_ABSENT`, otherwise `CONTINUE_MODE_A_SEQUENCE`. It is **not** a gate:
every scale build still finishes.

In [1]:
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

import numpy as np
import pandas as pd

from casmi.paths import get_project_paths
from casmi.qcr.context import HOST_INGEST_LIB, v4b_paths, v5_paths
from casmi.qcr.settlement import load_settlement_or_refuse
from casmi.ranking.manifests import load_manifest, save_manifest
from casmi.ranking.rank_eval import summarize
from casmi.ranking.scaling import pool_size_stratification
from casmi.ranking.training_sets import MODE_A_SUFFIX, derive_mode_a_manifests
from casmi.validation.regime_audit import (
    AUDIT_PROTOCOLS, INCONSISTENT, MODE_A_MIRROR, NOT_BUILT, REF_ABSENT, REGIMES, ROUTE_REF_ABSENT, ROUTE_THRESHOLD, ArtifactNotBuilt,
    build_query_regimes, load_host_truth_evidence, load_v5_truth_evidence, metrics_by_regime, molecule_regime_mix, project_route,
    save_regimes, source_breakdown, summary_row,
)

pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220)
paths = get_project_paths()
vp, D = v4b_paths(paths), v5_paths(paths)
REG_DIR = D['root'] / 'regimes'
REG_DIR.mkdir(parents=True, exist_ok=True)
settlement = load_settlement_or_refuse(vp.settlement_json)
ARTS = settlement['evidence_artifacts']
UNITS = {'TL_EVAL': ['TL_EVAL'], 'TL_1K': ['TL_1K'], 'TL_3K': ['TL_1K', 'TL_3K'], 'TL_10K': ['TL_1K', 'TL_3K', 'TL_10K'],
         'RND_1K': ['RND_1K'], 'MOL_DEV': ['MOL_DEV']}
DATASETS = ['TL_EVAL', 'HOST', 'TL_1K', 'TL_3K', 'TL_10K', 'RND_1K', 'MOL_DEV']
print('protocols audited:', AUDIT_PROTOCOLS, '| routing threshold (pre-registered):', ROUTE_THRESHOLD)

protocols audited: ('standard', 'mirror_aware') | routing threshold (pre-registered): 0.5


## 1. Query-level regimes per dataset (existing QCR only; `NOT_BUILT` when absent)

In [2]:
REG, STATUS = {}, {}
host_q = pd.read_parquet(vp.host_processed / 'host_holdout_spectra.parquet')
host_manifest = host_q.rename(columns={'true_connectivity_key': 'connectivity_key', 'ingest_lib': 'source', 'instrument_type': 'instrument'})
for name in DATASETS:
    try:
        if name == 'HOST':
            manifest = host_manifest
            pool, qcr_t, pairs_t = load_host_truth_evidence(ARTS)
        else:
            if not (D['manifests'] / f'{name}.parquet').exists():
                raise ArtifactNotBuilt(f'manifest {name} missing')
            manifest, _ = load_manifest(name, D['manifests'])
            pool, qcr_t, pairs_t = load_v5_truth_evidence(D, UNITS[name])
            pool = pool[pool['query_id'].isin(set(manifest['query_id']))]
        r = build_query_regimes(manifest, pool, qcr_t, pairs_t)
        assert len(r) == len(manifest) and not r['query_id'].duplicated().any()
        assert set(r['regime']) <= set(REGIMES)
        assert (r['regime'] == INCONSISTENT).sum() == 0, 'INCONSISTENT regime rows: standard must include every mirror_aware reference'
        meta = save_regimes(r, name, REG_DIR)
        REG[name], STATUS[name] = r, 'BUILT'
        print(f'{name}: {len(r):,} queries -> {meta["regime_counts"]}')
    except (ArtifactNotBuilt, FileNotFoundError) as e:
        STATUS[name] = NOT_BUILT
        print(f'{name}: {NOT_BUILT} ({e})')
print(STATUS)

c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\qcr\aggregate.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["exhausted"] = out["exhausted"].fillna(True).astype(bool)
c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\qcr\aggregate.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["exhausted"] = out["exhausted"].fillna(True).astype(bool)


TL_EVAL: 1,000 queries -> {'STANDARD_ONLY': 984, 'MODE_A_MIRROR': 14, 'REF_ABSENT': 2}
HOST: 1,184 queries -> {'MODE_A_MIRROR': 1184}
TL_1K: 1,000 queries -> {'STANDARD_ONLY': 982, 'MODE_A_MIRROR': 18}
TL_3K: NOT_BUILT (unit TL_3K: QCR shards / pool not found under C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\qcr\TL_3K)
TL_10K: NOT_BUILT (unit TL_3K: QCR shards / pool not found under C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\qcr\TL_3K)
RND_1K: 1,000 queries -> {'STANDARD_ONLY': 643, 'MODE_A_MIRROR': 335, 'POOL_ABSENT': 13, 'REF_ABSENT': 9}
MOL_DEV: NOT_BUILT (unit MOL_DEV: QCR shards / pool not found under C:\Users\myben\OneDrive\Documents\Enveda\outputs\v5\qcr\MOL_DEV)
{'TL_EVAL': 'BUILT', 'HOST': 'BUILT', 'TL_1K': 'BUILT', 'TL_3K': 'NOT_BUILT', 'TL_10K': 'NOT_BUILT', 'RND_1K': 'BUILT', 'MOL_DEV': 'NOT_BUILT'}


c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\qcr\aggregate.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["exhausted"] = out["exhausted"].fillna(True).astype(bool)
c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\qcr\aggregate.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  out["exhausted"] = out["exhausted"].fillna(True).astype(bool)
c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\validation\regime_audit.py:87: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future 

## 2. Primary summary table (percent of each dataset's queries)

In [3]:
rows = []
for name in DATASETS:
    if STATUS[name] != 'BUILT':
        rows.append({'dataset': name, 'status': NOT_BUILT})
        continue
    s = summary_row(REG[name])
    rows.append({'dataset': name, 'status': 'BUILT', 'n_queries': s['n_queries'], 'truth_in_pool_pct': 100 * s['truth_in_pool_share'],
                 **{f'{p}_ge{k}_pct': 100 * s[f'{p}_ge{k}_share'] for p in ('standard', 'mirror') for k in (1, 3, 5)},
                 'MODE_A_MIRROR_pct': 100 * s['mode_a_share'], 'STANDARD_ONLY_pct': 100 * s['standard_only_share'],
                 'REF_ABSENT_pct': 100 * s['ref_absent_share'], 'POOL_ABSENT_pct': 100 * s['pool_absent_share'],
                 'median_mirror_refs_censored': s['median_mirror_refs_censored']})
summary_table = pd.DataFrame(rows).set_index('dataset')
display(summary_table.round(2))
summary_table.to_csv(REG_DIR / 'regime_summary_table.csv')

,status,n_queries,truth_in_pool_pct,standard_ge1_pct,standard_ge3_pct,standard_ge5_pct,mirror_ge1_pct,mirror_ge3_pct,mirror_ge5_pct,MODE_A_MIRROR_pct,STANDARD_ONLY_pct,REF_ABSENT_pct,POOL_ABSENT_pct,median_mirror_refs_censored
dataset,,,,,,,,,,,,,,
TL_EVAL,BUILT,1000.0,100.0,99.8,89.1,60.00,1.4,1.40,1.40,1.4,98.4,0.2,0.0,0.0
HOST,BUILT,1184.0,100.0,100.0,100.0,99.75,100.0,99.32,98.99,100.0,0.0,0.0,0.0,>=5
TL_1K,BUILT,1000.0,100.0,100.0,87.2,60.20,1.8,1.60,1.60,1.8,98.2,0.0,0.0,0.0
TL_3K,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TL_10K,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
RND_1K,BUILT,1000.0,98.7,97.8,92.1,80.40,33.5,29.20,26.80,33.5,64.3,0.9,1.3,0.0
MOL_DEV,NOT_BUILT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Source-library breakdown (the same-library-exclusion question)

In [4]:
SRC = {}
for name in DATASETS:
    if STATUS[name] == 'BUILT':
        SRC[name] = source_breakdown(REG[name])
        SRC[name].to_csv(REG_DIR / f'{name}_source_breakdown.csv', index=False)
        print(f'\n== {name}')
        cols = ['source', 'n_queries', 'truth_in_pool_share', 'standard_ge1_share', 'standard_ge3_share', 'standard_ge5_share', 'mirror_ge1_share',
                'mirror_ge3_share', 'mirror_ge5_share', 'mode_a_share', 'standard_only_share', 'ref_absent_share', 'pool_absent_share']
        display(SRC[name][cols].sort_values('n_queries', ascending=False).round(3))


== TL_EVAL


,source,n_queries,truth_in_pool_share,standard_ge1_share,standard_ge3_share,standard_ge5_share,mirror_ge1_share,mirror_ge3_share,mirror_ge5_share,mode_a_share,standard_only_share,ref_absent_share,pool_absent_share
0,enveda-180,1000,1.0,0.998,0.891,0.6,0.014,0.014,0.014,0.014,0.984,0.002,0.0



== HOST


c:\Users\myben\OneDrive\Documents\Enveda\src\casmi\validation\regime_audit.py:171: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for src, g in regimes.groupby(source_col, dropna=False, sort=True):


,source,n_queries,truth_in_pool_share,standard_ge1_share,standard_ge3_share,standard_ge5_share,mirror_ge1_share,mirror_ge3_share,mirror_ge5_share,mode_a_share,standard_only_share,ref_absent_share,pool_absent_share
2,enveda-np-examples,1184,1.0,1.0,1.0,0.997,1.0,0.993,0.99,1.0,0.0,0.0,0.0
0,drug_plus,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,enveda-180,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gnps,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,masaryk,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,massbank,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,mona,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,msdial,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,pluskal_ms2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,riken,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



== TL_1K


,source,n_queries,truth_in_pool_share,standard_ge1_share,standard_ge3_share,standard_ge5_share,mirror_ge1_share,mirror_ge3_share,mirror_ge5_share,mode_a_share,standard_only_share,ref_absent_share,pool_absent_share
0,enveda-180,981,1.0,1.0,0.87,0.595,0.010,0.01,0.01,0.010,0.990,0.0,0.0
5,pluskal_ms2,12,1.0,1.0,1.00,0.917,0.167,0.00,0.00,0.167,0.833,0.0,0.0
2,massbank,2,1.0,1.0,1.00,1.000,1.000,1.00,1.00,1.000,0.000,0.0,0.0
6,riken,2,1.0,1.0,1.00,1.000,0.500,0.50,0.50,0.500,0.500,0.0,0.0
1,gnps,1,1.0,1.0,1.00,1.000,1.000,1.00,1.00,1.000,0.000,0.0,0.0
3,mona,1,1.0,1.0,1.00,1.000,1.000,1.00,1.00,1.000,0.000,0.0,0.0
4,msdial,1,1.0,1.0,1.00,1.000,1.000,1.00,1.00,1.000,0.000,0.0,0.0



== RND_1K


,source,n_queries,truth_in_pool_share,standard_ge1_share,standard_ge3_share,standard_ge5_share,mirror_ge1_share,mirror_ge3_share,mirror_ge5_share,mode_a_share,standard_only_share,ref_absent_share,pool_absent_share
1,enveda-180,467,1.000,1.000,0.912,0.711,0.004,0.004,0.004,0.004,0.996,0.000,0.000
6,pluskal_ms2,206,1.000,1.000,0.995,0.908,0.447,0.262,0.199,0.447,0.553,0.000,0.000
7,riken,117,0.949,0.949,0.949,0.949,0.812,0.812,0.778,0.812,0.137,0.000,0.051
2,gnps,83,0.964,0.904,0.771,0.735,0.639,0.639,0.602,0.639,0.265,0.060,0.036
3,massbank,42,0.929,0.929,0.881,0.881,0.810,0.786,0.762,0.810,0.119,0.000,0.071
4,mona,40,0.975,0.975,0.925,0.925,0.925,0.875,0.875,0.925,0.050,0.000,0.025
8,spectraverse,32,1.000,0.875,0.875,0.844,0.281,0.250,0.188,0.281,0.594,0.125,0.000
5,msdial,11,1.000,1.000,1.000,0.909,1.000,0.909,0.818,1.000,0.000,0.000,0.000
0,drug_plus,2,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000


## 4. Standard vs mirror_aware diagnostic (TL_EVAL, HOST). Diagnostic only; no protocol switch, no selection.

This uses the V1-compatible spectral-feature coverage: whether a candidate has at least one
reference selected by the protocol, i.e. whether its 8 spectral features are finite. It is
measured for truth and decoy candidates, from the existing QCR `accepted_<protocol>` flags.

In [5]:
def coverage_rows(name, qcr_paths, pool):
    parts = []
    for p in qcr_paths:
        q = pd.read_parquet(p, columns=['query_id', 'candidate_key', 'accepted_standard', 'accepted_mirror_aware'])
        parts.append(q.groupby(['query_id', 'candidate_key'])[['accepted_standard', 'accepted_mirror_aware']].any().reset_index())
    cov = pd.concat(parts, ignore_index=True).groupby(['query_id', 'candidate_key']).any().reset_index()
    d = pool.merge(cov.rename(columns={'candidate_key': 'candidate_connectivity_key'}), on=['query_id', 'candidate_connectivity_key'], how='left')
    d[['accepted_standard', 'accepted_mirror_aware']] = d[['accepted_standard', 'accepted_mirror_aware']].fillna(False).astype(bool)
    out = []
    for role, g in (('truth', d[d['is_true_candidate']]), ('decoy', d[~d['is_true_candidate']])):
        out.append({'dataset': name, 'candidate_role': role, 'n_candidates': int(len(g)),
                    'coverage_standard': float(g['accepted_standard'].mean()), 'coverage_mirror_aware': float(g['accepted_mirror_aware'].mean()),
                    'available_to_absent_share': float((g['accepted_standard'] & ~g['accepted_mirror_aware']).mean())})
    return out


cov_rows = []
if STATUS['TL_EVAL'] == 'BUILT':
    cov_rows += coverage_rows('TL_EVAL', sorted((D['qcr'] / 'TL_EVAL' / 'qcr' / 'shards').glob('part-*.parquet')),
                              pd.read_parquet(D['features'] / 'units' / 'TL_EVAL_pool.parquet'))
cov_rows += coverage_rows('HOST', [ARTS['host_qcr']], pd.read_parquet(ARTS['host_features_mirror_aware'], columns=['query_id', 'candidate_connectivity_key', 'is_true_candidate']))
coverage = pd.DataFrame(cov_rows)
coverage.to_csv(REG_DIR / 'standard_vs_mirror_coverage.csv', index=False)
display(coverage.round(4))

C:\Users\myben\AppData\Local\Temp\ipykernel_12888\2308238768.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  d[['accepted_standard', 'accepted_mirror_aware']] = d[['accepted_standard', 'accepted_mirror_aware']].fillna(False).astype(bool)


,dataset,candidate_role,n_candidates,coverage_standard,coverage_mirror_aware,available_to_absent_share
0,TL_EVAL,truth,1000,0.998,0.0140,0.9840
1,TL_EVAL,decoy,358875,1.000,0.1831,0.8169
2,HOST,truth,1184,1.000,1.0000,0.0000
3,HOST,decoy,125389,1.000,1.0000,0.0000


## 5. TL_EVAL performance by regime (existing provisional 1k predictions; nothing trained or tuned)

Rows show all queries (`ALL`, with pool-absent queries at RR = 0) and then each regime. V1 is
compared with B1 (mass only), which separates missing reference evidence from generally dense
pools. B5 is the frozen v4b configuration, with nothing tuned on TL_EVAL.

In [6]:
PERF = {}
if STATUS['TL_EVAL'] == 'BUILT':
    for label, fname in (('V1_TL_1K', 'tl_eval_per_query_V1_TL_1K'), ('V1_RND_1K', 'tl_eval_per_query_V1_RND_1K'),
                         ('B1_mass', 'tl_eval_per_query_B1_mass'), ('B5_frozen_v4b', 'tl_eval_per_query_B5_frozen_v4b')):
        p = D['metrics'] / f'{fname}.parquet'
        if p.exists():
            PERF[label] = metrics_by_regime(pd.read_parquet(p), REG['TL_EVAL']).assign(model=label)
        else:
            print(f'{label}: prediction table not found ({p.name}) -- run 10v5_02 at 1k first')
    perf = pd.concat(PERF.values(), ignore_index=True) if PERF else pd.DataFrame()
    if len(perf):
        display(perf.pivot_table(index=['population'], columns='model', values='mrr_at_25').round(4))
        display(perf[['model', 'population', 'n_queries', 'mrr_at_25', 'hit_at_1', 'hit_at_5', 'hit_at_25']].round(4))
        perf.to_csv(REG_DIR / 'tl_eval_metrics_by_regime.csv', index=False)

model,B1_mass,B5_frozen_v4b,V1_RND_1K,V1_TL_1K
population,,,,
ALL,0.1418,0.1274,0.1836,0.1866
MODE_A_MIRROR,0.2129,0.9286,0.7143,0.6506
REF_ABSENT,0.1250,0.1000,0.0000,0.1667
STANDARD_ONLY,0.1408,0.1161,0.1765,0.1801


,model,population,n_queries,mrr_at_25,hit_at_1,hit_at_5,hit_at_25
0,V1_TL_1K,ALL,1000,0.1866,0.0930,0.2720,0.6560
1,V1_TL_1K,MODE_A_MIRROR,14,0.6506,0.6429,0.6429,0.7857
2,V1_TL_1K,STANDARD_ONLY,984,0.1801,0.0854,0.2663,0.6545
3,V1_TL_1K,REF_ABSENT,2,0.1667,0.0000,0.5000,0.5000
4,V1_TL_1K,POOL_ABSENT,0,NaN,NaN,NaN,NaN
5,V1_RND_1K,ALL,1000,0.1836,0.0900,0.2730,0.6410
6,V1_RND_1K,MODE_A_MIRROR,14,0.7143,0.7143,0.7143,0.7143
7,V1_RND_1K,STANDARD_ONLY,984,0.1765,0.0813,0.2673,0.6413
8,V1_RND_1K,REF_ABSENT,2,0.0000,0.0000,0.0000,0.0000
9,V1_RND_1K,POOL_ABSENT,0,NaN,NaN,NaN,NaN


## 6. Candidate density by regime (TL_EVAL; n shown)

In [7]:
if STATUS['TL_EVAL'] == 'BUILT' and (D['metrics'] / 'tl_eval_per_query_V1_TL_1K.parquet').exists():
    pq = pd.read_parquet(D['metrics'] / 'tl_eval_per_query_V1_TL_1K.parquet').merge(REG['TL_EVAL'][['query_id', 'regime']], on='query_id')
    dens = pd.concat([pool_size_stratification(pq[pq['regime'] == g]).assign(regime=g) for g in (MODE_A_MIRROR, REF_ABSENT)]
                     + [pool_size_stratification(pq).assign(regime='ALL')], ignore_index=True)
    display(dens)
    dens.to_csv(REG_DIR / 'tl_eval_density_by_regime_V1_TL_1K.csv', index=False)

,pool_bin,n_queries,mrr,hit_at_1,median_pool,low_n,regime
0,0-49,1,1.000000,1.000000,33.0,True,MODE_A_MIRROR
1,50-99,1,1.000000,1.000000,62.0,True,MODE_A_MIRROR
2,100-199,4,0.516667,0.500000,161.5,True,MODE_A_MIRROR
3,200-399,4,0.750000,0.750000,270.0,True,MODE_A_MIRROR
4,400-799,4,0.510417,0.500000,551.0,True,MODE_A_MIRROR
5,>=800,0,NaN,NaN,NaN,True,MODE_A_MIRROR
6,0-49,0,NaN,NaN,NaN,True,REF_ABSENT
7,50-99,0,NaN,NaN,NaN,True,REF_ABSENT
8,100-199,0,NaN,NaN,NaN,True,REF_ABSENT
9,200-399,1,0.000000,0.000000,382.0,True,REF_ABSENT


## 7. Mode-A training manifests (derived; the originals are never modified)

A training query is kept iff its truth is in the pool and has at least one eligible
`mirror_aware` reference. The filter is a per-query decision, so
`TL_1K_MODE_A ⊂ TL_3K_MODE_A ⊂ TL_10K_MODE_A` is asserted and never repaired by resampling.
No QCR is rebuilt. No regime, reference-count or availability value is ever a model feature.

In [8]:
train_names = [n for n in ('TL_1K', 'TL_3K', 'TL_10K', 'RND_1K') if STATUS.get(n) == 'BUILT']
originals = {n: load_manifest(n, D['manifests'])[0] for n in train_names}
before_sha = {n: load_manifest(n, D['manifests'])[1]['manifest_sha256'] for n in train_names}
mode_a, audits = derive_mode_a_manifests(originals, {n: REG[n] for n in train_names})
tl_keys, host_keys = set(load_manifest('TL_EVAL', D['manifests'])[0]['connectivity_key']), set(host_q['true_connectivity_key'])
for name, m in mode_a.items():
    assert (m['query_id'].map(REG[name.replace(MODE_A_SUFFIX, '')].set_index('query_id')['regime']) == MODE_A_MIRROR).all()
    assert not (set(m['connectivity_key']) & tl_keys) and not (set(m['connectivity_key']) & host_keys)
    assert HOST_INGEST_LIB not in set(m['source'])
    save_manifest(m, name, D['manifests'], extra={'derived_from': name.replace(MODE_A_SUFFIX, ''), **audits[name]})
after_sha = {n: load_manifest(n, D['manifests'])[1]['manifest_sha256'] for n in train_names}
assert before_sha == after_sha, 'an original manifest changed'
retention = pd.DataFrame(audits).T[['n_original', 'n_mode_a_retained', 'retention_rate']]
display(retention)

# missingness-shortcut diagnostic (audit only): every retained training query's truth has a mirror reference;
# decoy no-reference rates are reported, never used as a feature
diag = []
for name in mode_a:
    base = name.replace(MODE_A_SUFFIX, '')
    fp = D['features'] / f'{base}_mirror_aware.parquet'
    if not fp.exists():
        continue
    f = pd.read_parquet(fp, columns=['query_id', 'is_true_candidate', 'n_reference_spectra'])
    f = f[f['query_id'].isin(set(mode_a[name]['query_id']))]
    truth_ok = f[f['is_true_candidate']].groupby('query_id')['n_reference_spectra'].min().ge(1)
    assert truth_ok.all() and truth_ok.size == mode_a[name]['query_id'].nunique(), f'{name}: a retained truth lacks a mirror_aware reference'
    diag.append({'manifest': name, 'n_queries': int(truth_ok.size), 'truth_has_mirror_reference_share': float(truth_ok.mean()),
                 'decoy_no_reference_rate': float((f.loc[~f['is_true_candidate'], 'n_reference_spectra'] == 0).mean())})
if diag:
    display(pd.DataFrame(diag))

,n_original,n_mode_a_retained,retention_rate
TL_1K_MODE_A,1000,18,0.018
RND_1K_MODE_A,1000,335,0.335


,manifest,n_queries,truth_has_mirror_reference_share,decoy_no_reference_rate
0,TL_1K_MODE_A,18,1.0,0.533158
1,RND_1K_MODE_A,335,1.0,0.068339


## 8. MOL_DEV molecule regime mix (preparation for aggregation; not used for any selection)

In [9]:
if STATUS.get('MOL_DEV') == 'BUILT':
    mix = molecule_regime_mix(REG['MOL_DEV'])
    mix.to_parquet(REG_DIR / 'MOL_DEV_molecule_regime_mix.parquet', index=False)
    display(mix['molecule_regime_mix'].value_counts().rename('n_molecules').to_frame())
else:
    print('MOL_DEV: NOT_BUILT -- rerun this notebook after scripts/v5_build_scale_features.py --manifest MOL_DEV')

MOL_DEV: NOT_BUILT -- rerun this notebook after scripts/v5_build_scale_features.py --manifest MOL_DEV


## 9. Summary JSON, project route and decision text (all computed at runtime)

In [10]:
def summary_fields(name):
    if STATUS.get(name) != 'BUILT':
        return {'status': NOT_BUILT}
    s = summary_row(REG[name])
    keys = ['n_queries', 'truth_in_pool_share', 'standard_ge1_share', 'mirror_ge1_share', 'mirror_ge3_share', 'mirror_ge5_share', 'mode_a_share',
            'standard_only_share', 'ref_absent_share', 'pool_absent_share', 'median_mirror_refs_censored']
    return {'status': 'BUILT', **{k: s[k] for k in keys}}


route = project_route(summary_row(REG['TL_EVAL'])['mode_a_share']) if STATUS['TL_EVAL'] == 'BUILT' else {'project_route': NOT_BUILT}
model_metrics = {}
if 'V1_TL_1K' in PERF:
    p = PERF['V1_TL_1K'].set_index('population')
    model_metrics = {f'TL_EVAL_{k}': p.loc[v].drop('model').to_dict() for k, v in
                     (('ALL', 'ALL'), ('MODE_A', MODE_A_MIRROR), ('STANDARD_ONLY', 'STANDARD_ONLY'), ('REF_ABSENT', REF_ABSENT)) if v in p.index}
audit_summary = {'status': STATUS, **{n: summary_fields(n) for n in DATASETS},
                 'training': {n: audits[n] for n in audits}, 'model_metrics_V1_TL_1K_provisional': model_metrics,
                 'coverage_standard_vs_mirror': cov_rows, 'project_route': route['project_route']}
(D['root'] / 'regime_audit_summary.json').write_text(json.dumps(audit_summary, indent=2, default=str), encoding='utf-8')
(D['root'] / 'project_route.json').write_text(json.dumps(route, indent=2, default=str), encoding='utf-8')

if STATUS['TL_EVAL'] == 'BUILT':
    s = summary_row(REG['TL_EVAL'])
    print(f"TL-EVAL mirror-aware Mode-A coverage: {100 * s['mode_a_share']:.1f}%")
    print(f"TL-EVAL standard-only coverage:       {100 * s['standard_only_share']:.1f}%")
    print(f"TL-EVAL fully reference-absent:       {100 * s['ref_absent_share']:.1f}%")
    print(f"TL-EVAL truth absent from pool:       {100 * s['pool_absent_share']:.1f}%")
    print(f"\nPROJECT ROUTE: {route['project_route']}  ({route['meaning']})")
    if route['project_route'] == ROUTE_REF_ABSENT:
        print('Next research notebook after the freeze: reference-absent route -- see docs/ROADMAP_REFERENCE_ABSENT.md')

TL-EVAL mirror-aware Mode-A coverage: 1.4%
TL-EVAL standard-only coverage:       98.4%
TL-EVAL fully reference-absent:       0.2%
TL-EVAL truth absent from pool:       0.0%

PROJECT ROUTE: PRIORITIZE_REFERENCE_ABSENT  (finish TL_3K/TL_10K builds and the Mode-A scale comparison; the NEXT major research effort is reference-absent / Mode-B evidence)
Next research notebook after the freeze: reference-absent route -- see docs/ROADMAP_REFERENCE_ABSENT.md
